# 1. Introducci?n

No todos los tiros son iguales. Un disparo desde el punto de penalti, sin presi?n y con el portero descolocado, no puede valorarse igual que un remate forzado desde la banda con tres defensores encima. En f?tbol, el marcador solo nos dice qu? ocurri?; el Expected Goals (xG) intenta medir qu? tan probable era que ocurriera.

Este proyecto construye un modelo propio de xG a partir de eventos de tiros de StatsBomb Open Data. La idea no es solo entrenar un clasificador, sino contar una historia completa: c?mo convertir coordenadas y contexto futbol?stico en probabilidades calibradas, c?mo comparar el modelo con el xG de StatsBomb y c?mo extraer conclusiones ?tiles para an?lisis deportivo.

# 2. Objetivo del proyecto

El objetivo principal es predecir la probabilidad de que un tiro termine en gol.

Objetivos espec?ficos:

- Preparar un dataset de tiros usando una estructura compatible con StatsBomb Open Data.
- Dise?ar variables futbol?sticas interpretables: distancia, ?ngulo, parte del cuerpo, tipo de jugada, presi?n y zona del campo.
- Crear una variable propia de `big_chance` y una m?trica de eficiencia ofensiva.
- Entrenar y comparar Logistic Regression, Random Forest y XGBoost.
- Evaluar rendimiento con ROC-AUC, Brier Score y curvas de calibraci?n.
- Comparar el modelo propio con el xG de StatsBomb.
- Comunicar insights de forma clara para un contexto de m?ster.

# 3. Descripci?n del dataset

StatsBomb Open Data publica eventos de partidos en formato JSON. Cada evento contiene informaci?n como:

- Tipo de evento: pase, tiro, duelo, conducci?n, etc.
- Equipo, jugador y minuto.
- Coordenadas del evento en un campo de 120x80.
- Para tiros: resultado, parte del cuerpo, tipo de jugada, localizaci?n final, xG de StatsBomb y contexto.

En una ejecuci?n real, este notebook leer?a los JSON de `open-data/data/events/`. Para que el notebook sea reproducible sin descargar datos, el c?digo incluye un fallback que simula un dataset de tiros con la misma l?gica de columnas.

In [ ]:
# Imports principales
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import calibration_curve

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

try:
    from mplsoccer import Pitch
    HAS_MPLSOCCER = True
except Exception:
    HAS_MPLSOCCER = False

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

print(f"mplsoccer disponible: {HAS_MPLSOCCER}")
print(f"xgboost disponible: {HAS_XGBOOST}")

# 4. Carga y limpieza de datos

La funci?n de carga busca ficheros JSON de eventos. Si existen, extrae tiros reales de StatsBomb. Si no existen, genera datos sint?ticos realistas: tiros m?s cercanos tienen mayor probabilidad de gol, la presi?n reduce la calidad del tiro y el pie/cabeza/tipo de jugada modifican la probabilidad.

Este enfoque permite entregar un notebook ejecutable y, al mismo tiempo, dejar preparada la conexi?n con datos reales.

In [ ]:
def simulate_shots(n_shots: int = 4500) -> pd.DataFrame:
    teams = ["Barcelona", "Real Madrid", "Atletico", "Valencia", "Sevilla", "Real Sociedad"]
    players = [
        "Alex Martin", "Pablo Ruiz", "Dani Torres", "Hugo Silva", "Marcos Leon",
        "Ivan Costa", "Ruben Vidal", "Sergio Molina", "Adrian Vega", "Nico Santos"
    ]
    body_parts = ["Right Foot", "Left Foot", "Head", "Other"]
    shot_types = ["Open Play", "Free Kick", "Corner", "Penalty", "Fast Break"]

    x = np.clip(np.random.beta(5.5, 2.2, n_shots) * 120, 0, 120)
    y = np.clip(np.random.normal(40, 17, n_shots), 0, 80)
    body = np.random.choice(body_parts, n_shots, p=[0.48, 0.32, 0.17, 0.03])
    shot_type = np.random.choice(shot_types, n_shots, p=[0.75, 0.07, 0.08, 0.04, 0.06])
    under_pressure = np.random.choice([0, 1], n_shots, p=[0.68, 0.32])

    goal_x, goal_y = 120, 40
    distance = np.sqrt((goal_x - x) ** 2 + (goal_y - y) ** 2)
    goal_width = 7.32 / 68 * 80
    left_post_y = goal_y - goal_width / 2
    right_post_y = goal_y + goal_width / 2
    angle = np.abs(
        np.arctan2(right_post_y - y, goal_x - x) - np.arctan2(left_post_y - y, goal_x - x)
    )

    logit = (
        2.8
        - 0.15 * distance
        + 1.35 * angle
        - 0.55 * under_pressure
        + np.where(body == "Head", -0.35, 0)
        + np.where(shot_type == "Penalty", 2.5, 0)
        + np.where(shot_type == "Fast Break", 0.45, 0)
        + np.where(shot_type == "Free Kick", -0.55, 0)
    )
    true_prob = 1 / (1 + np.exp(-logit))
    goal = np.random.binomial(1, true_prob)
    statsbomb_xg = np.clip(true_prob + np.random.normal(0, 0.035, n_shots), 0.01, 0.92)

    return pd.DataFrame({
        "match_id": np.random.randint(1, 120, n_shots),
        "team": np.random.choice(teams, n_shots),
        "player": np.random.choice(players, n_shots),
        "minute": np.random.randint(1, 96, n_shots),
        "location_x": x,
        "location_y": y,
        "shot_body_part": body,
        "shot_type": shot_type,
        "under_pressure": under_pressure.astype(bool),
        "goal": goal,
        "outcome": np.where(goal == 1, "Goal", "No Goal"),
        "statsbomb_xg": statsbomb_xg,
        "data_source": "simulated_statsbomb_like"
    })


def load_statsbomb_shots(events_dir: Path) -> pd.DataFrame:
    event_files = sorted(events_dir.glob("*.json"))
    if not event_files:
        print("No se encontraron eventos reales. Se usa dataset simulado compatible con StatsBomb.")
        return simulate_shots()

    rows = []
    for file in event_files:
        with open(file, "r", encoding="utf-8") as f:
            events = json.load(f)
        for ev in events:
            if ev.get("type", {}).get("name") != "Shot":
                continue
            shot = ev.get("shot", {})
            loc = ev.get("location", [np.nan, np.nan])
            rows.append({
                "match_id": file.stem,
                "team": ev.get("team", {}).get("name"),
                "player": ev.get("player", {}).get("name"),
                "minute": ev.get("minute"),
                "location_x": loc[0] if len(loc) > 0 else np.nan,
                "location_y": loc[1] if len(loc) > 1 else np.nan,
                "shot_body_part": shot.get("body_part", {}).get("name"),
                "shot_type": shot.get("type", {}).get("name"),
                "under_pressure": bool(ev.get("under_pressure", False)),
                "outcome": shot.get("outcome", {}).get("name"),
                "goal": int(shot.get("outcome", {}).get("name") == "Goal"),
                "statsbomb_xg": shot.get("statsbomb_xg"),
                "data_source": "statsbomb_open_data"
            })

    shots = pd.DataFrame(rows)
    print(f"Tiros cargados desde StatsBomb Open Data: {len(shots):,}")
    return shots

EVENTS_DIR = Path("../data/statsbomb/open-data/data/events")
shots_raw = load_statsbomb_shots(EVENTS_DIR)
shots_raw.head()

In [ ]:
shots = shots_raw.copy()

required = ["location_x", "location_y", "shot_body_part", "shot_type", "goal"]
shots = shots.dropna(subset=required)
shots["statsbomb_xg"] = shots["statsbomb_xg"].fillna(shots["goal"].mean())
shots["under_pressure"] = shots["under_pressure"].fillna(False).astype(bool)
shots["player"] = shots["player"].fillna("Unknown player")
shots["team"] = shots["team"].fillna("Unknown team")

print(shots.shape)
shots.describe(include="all").T.head(12)

# 5. An?lisis exploratorio (EDA)

Antes de entrenar modelos conviene mirar el f?tbol que hay detr?s de la tabla. El EDA busca responder preguntas b?sicas:

- ?Desde d?nde se tira?
- ?C?mo cambia la tasa de gol con la distancia?
- ?Qu? jugadores acumulan m?s xG y goles?
- ?La distribuci?n parece compatible con un problema probabil?stico bien planteado?

In [ ]:
def draw_pitch(ax=None):
    if HAS_MPLSOCCER:
        pitch = Pitch(pitch_type="statsbomb", pitch_color="#fbfcfd", line_color="#94a3b8")
        fig, ax = pitch.draw(ax=ax)
        return fig, ax
    if ax is None:
        fig, ax = plt.subplots(figsize=(11, 7))
    else:
        fig = ax.figure
    ax.set_xlim(0, 120)
    ax.set_ylim(0, 80)
    ax.set_aspect("equal")
    ax.plot([0, 120, 120, 0, 0], [0, 0, 80, 80, 0], color="#94a3b8")
    ax.plot([60, 60], [0, 80], color="#94a3b8")
    ax.add_patch(plt.Rectangle((102, 18), 18, 44, fill=False, color="#94a3b8"))
    ax.add_patch(plt.Rectangle((0, 18), 18, 44, fill=False, color="#94a3b8"))
    ax.set_xticks([])
    ax.set_yticks([])
    return fig, ax

fig, ax = plt.subplots(figsize=(11, 7))
draw_pitch(ax)
sample = shots.sample(min(1200, len(shots)), random_state=RANDOM_STATE)
colors = np.where(sample["goal"] == 1, "#0f172a", "#38bdf8")
ax.scatter(sample["location_x"], sample["location_y"], c=colors, alpha=0.55, s=22, edgecolor="white", linewidth=0.2)
ax.set_title("Mapa de tiros: goles frente a no goles")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=shots, x="outcome", ax=axes[0], palette=["#38bdf8", "#0f172a"])
axes[0].set_title("Goles vs no goles")
axes[0].set_xlabel("")
axes[0].set_ylabel("Tiros")

sns.histplot(data=shots, x="statsbomb_xg", hue="goal", bins=35, kde=True, ax=axes[1], palette=["#38bdf8", "#0f172a"])
axes[1].set_title("Distribuci?n del xG de StatsBomb por resultado")
axes[1].set_xlabel("StatsBomb xG")
plt.tight_layout()
plt.show()

Conclusi?n intermedia: el problema tiene una fuerte descompensaci?n natural. La mayor?a de tiros no son gol, por lo que la calibraci?n ser? tan importante como la capacidad de ordenar tiros peligrosos.

# 6. Feature engineering

El valor diferencial del proyecto est? en transformar eventos en lenguaje futbol?stico. Las variables elegidas son interpretables:

- `distance_to_goal`: distancia al centro de la porter?a.
- `shot_angle`: ?ngulo abierto respecto a los postes.
- `shot_zone`: zona del campo desde donde se dispara.
- `under_pressure`: si el tiro ocurre bajo presi?n.
- `big_chance_custom`: variable propia que identifica tiros cercanos, centrados y sin presi?n.
- `offensive_efficiency`: m?trica por jugador que compara goles reales con xG acumulado.

In [ ]:
def add_xg_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    goal_x, goal_y = 120, 40
    goal_width = 7.32 / 68 * 80
    left_post_y = goal_y - goal_width / 2
    right_post_y = goal_y + goal_width / 2

    data["distance_to_goal"] = np.sqrt((goal_x - data["location_x"]) ** 2 + (goal_y - data["location_y"]) ** 2)
    data["shot_angle"] = np.abs(
        np.arctan2(right_post_y - data["location_y"], goal_x - data["location_x"])
        - np.arctan2(left_post_y - data["location_y"], goal_x - data["location_x"])
    )
    data["centrality"] = np.abs(data["location_y"] - goal_y)

    conditions = [
        data["location_x"] >= 102,
        data["location_x"].between(90, 102),
        data["location_x"] < 90,
    ]
    choices = ["box", "edge_box", "long_range"]
    data["shot_zone"] = np.select(conditions, choices, default="unknown")

    data["big_chance_custom"] = (
        (data["distance_to_goal"] <= 16)
        & (data["centrality"] <= 12)
        & (~data["under_pressure"])
        & (data["shot_type"].isin(["Open Play", "Penalty", "Fast Break"]))
    ).astype(int)

    return data

shots_fe = add_xg_features(shots)
shots_fe[["distance_to_goal", "shot_angle", "shot_zone", "big_chance_custom", "goal"]].head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=shots_fe, x="distance_to_goal", hue="goal", bins=35, kde=True, ax=axes[0], palette=["#38bdf8", "#0f172a"])
axes[0].set_title("Distribuci?n de distancia a porter?a")
axes[0].set_xlabel("Distancia")

zone_summary = shots_fe.groupby("shot_zone").agg(shots=("goal", "size"), goal_rate=("goal", "mean"), avg_xg=("statsbomb_xg", "mean")).reset_index()
sns.barplot(data=zone_summary, x="shot_zone", y="goal_rate", ax=axes[1], color="#0f172a")
axes[1].set_title("Tasa de gol por zona")
axes[1].set_xlabel("Zona")
axes[1].set_ylabel("Tasa de gol")
plt.tight_layout()
plt.show()
zone_summary

In [ ]:
player_efficiency = (
    shots_fe.groupby("player")
    .agg(shots=("goal", "size"), goals=("goal", "sum"), xg=("statsbomb_xg", "sum"))
    .assign(offensive_efficiency=lambda d: d["goals"] - d["xg"])
    .query("shots >= 40")
    .sort_values("offensive_efficiency", ascending=False)
    .reset_index()
)

plt.figure(figsize=(10, 6))
top_players = player_efficiency.head(10)
sns.barplot(data=top_players, y="player", x="offensive_efficiency", color="#0f172a")
plt.axvline(0, color="#94a3b8", linewidth=1)
plt.title("Ranking de jugadores: goles - xG")
plt.xlabel("Eficiencia ofensiva")
plt.ylabel("")
plt.show()

top_players

# 7. Preparaci?n de datos para ML

Separamos variables num?ricas y categ?ricas. La regresi?n log?stica necesita escalado; los ?rboles no lo requieren, pero usar un `ColumnTransformer` permite mantener un pipeline coherente.

La variable objetivo es `goal`: 1 si el tiro termin? en gol, 0 en caso contrario.

In [ ]:
features_numeric = ["location_x", "location_y", "distance_to_goal", "shot_angle", "centrality", "big_chance_custom"]
features_categorical = ["shot_body_part", "shot_type", "shot_zone", "under_pressure"]
target = "goal"

X = shots_fe[features_numeric + features_categorical]
y = shots_fe[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), features_numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore"), features_categorical),
    ]
)

print(X_train.shape, X_test.shape)
print(f"Tasa de gol train: {y_train.mean():.3f} | test: {y_test.mean():.3f}")

# 8. Modelos

Se comparan tres familias:

- Logistic Regression: baseline interpretable y fuerte para probabilidades.
- Random Forest: captura no linealidades sin demasiada preparaci?n.
- XGBoost: suele funcionar muy bien en datos tabulares; se usa si est? instalado.

La m?trica principal no ser? solo ROC-AUC. En xG, predecir probabilidades bien calibradas es clave: un conjunto de tiros con 0.10 xG deber?a convertirse en gol aproximadamente el 10% de las veces.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("preprocess", preprocess),
        ("model", RandomForestClassifier(
            n_estimators=350,
            min_samples_leaf=20,
            max_depth=10,
            random_state=RANDOM_STATE,
            class_weight="balanced_subsample",
            n_jobs=-1
        ))
    ])
}

if HAS_XGBOOST:
    models["XGBoost"] = Pipeline([
        ("preprocess", preprocess),
        ("model", XGBClassifier(
            n_estimators=350,
            learning_rate=0.035,
            max_depth=3,
            subsample=0.85,
            colsample_bytree=0.85,
            eval_metric="logloss",
            random_state=RANDOM_STATE
        ))
    ])

results = []
predictions = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    predictions[name] = proba
    results.append({
        "model": name,
        "roc_auc": roc_auc_score(y_test, proba),
        "brier_score": brier_score_loss(y_test, proba),
        "mean_predicted_xg": proba.mean(),
    })

results_df = pd.DataFrame(results).sort_values(["roc_auc", "brier_score"], ascending=[False, True])
results_df

# 9. Evaluaci?n de modelos

ROC-AUC mide la capacidad de ordenar tiros peligrosos por encima de tiros menos peligrosos. Brier Score mide la calidad probabil?stica: penaliza probabilidades demasiado confiadas cuando se equivocan. La curva de calibraci?n permite ver si el modelo est? bien ajustado en rangos de probabilidad.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, proba in predictions.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[0].plot(fpr, tpr, label=f"{name} AUC={roc_auc_score(y_test, proba):.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="#94a3b8")
axes[0].set_title("Curvas ROC")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()

for name, proba in predictions.items():
    frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=10, strategy="quantile")
    axes[1].plot(mean_pred, frac_pos, marker="o", label=name)
axes[1].plot([0, 1], [0, 1], "--", color="#94a3b8")
axes[1].set_title("Curvas de calibraci?n")
axes[1].set_xlabel("Probabilidad predicha")
axes[1].set_ylabel("Frecuencia real de gol")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = models[best_model_name]
shots_test = shots_fe.loc[X_test.index].copy()
shots_test["model_xg"] = predictions[best_model_name]

print(f"Mejor modelo por ranking combinado: {best_model_name}")
print(results_df)

# 10. Interpretaci?n del modelo

En un proyecto de xG, el modelo debe ser ?til para explicar. Si el resultado depende de variables que no tienen sentido futbol?stico, el modelo puede estar explotando ruido. Por eso revisamos importancia de variables mediante permutation importance.

In [ ]:
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=8,
    random_state=RANDOM_STATE,
    scoring="roc_auc",
    n_jobs=-1,
)
importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, y="feature", x="importance_mean", color="#0f172a")
plt.title(f"Importancia por permutaci?n - {best_model_name}")
plt.xlabel("Ca?da media en ROC-AUC")
plt.ylabel("")
plt.show()
importance_df

In [ ]:
# Heatmap de probabilidad de gol: mantenemos constantes variables contextuales y movemos la posici?n del tiro.
grid = []
for gx in np.linspace(72, 120, 55):
    for gy in np.linspace(0, 80, 45):
        grid.append({
            "location_x": gx,
            "location_y": gy,
            "shot_body_part": "Right Foot",
            "shot_type": "Open Play",
            "under_pressure": False,
        })
grid_df = add_xg_features(pd.DataFrame(grid))
grid_df = grid_df[features_numeric + features_categorical]
grid_df["model_xg"] = best_model.predict_proba(grid_df)[:, 1]

fig, ax = plt.subplots(figsize=(11, 7))
draw_pitch(ax)
heat = ax.tricontourf(grid_df["location_x"], grid_df["location_y"], grid_df["model_xg"], levels=18, cmap="mako")
fig.colorbar(heat, ax=ax, label="Probabilidad de gol")
ax.set_title("Heatmap de probabilidad de gol estimada")
plt.show()

# 11. Comparaci?n con xG de StatsBomb

StatsBomb xG act?a como referencia externa. No buscamos demostrar que el modelo propio sea mejor, sino entender d?nde se aproxima, d?nde se separa y qu? puede aprenderse de esas diferencias.

In [ ]:
comparison = shots_test[["player", "team", "goal", "statsbomb_xg", "model_xg", "distance_to_goal", "shot_angle", "shot_type", "shot_body_part", "under_pressure"]].copy()
comparison["xg_diff_model_minus_statsbomb"] = comparison["model_xg"] - comparison["statsbomb_xg"]

statsbomb_metrics = {
    "model": "StatsBomb xG",
    "roc_auc": roc_auc_score(comparison["goal"], comparison["statsbomb_xg"]),
    "brier_score": brier_score_loss(comparison["goal"], comparison["statsbomb_xg"]),
    "mean_predicted_xg": comparison["statsbomb_xg"].mean(),
}

pd.concat([results_df, pd.DataFrame([statsbomb_metrics])], ignore_index=True).sort_values("brier_score")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=comparison.sample(min(900, len(comparison)), random_state=RANDOM_STATE), x="statsbomb_xg", y="model_xg", hue="goal", alpha=0.65, ax=axes[0], palette=["#38bdf8", "#0f172a"])
axes[0].plot([0, 1], [0, 1], "--", color="#94a3b8")
axes[0].set_title("Modelo propio vs StatsBomb xG")
axes[0].set_xlabel("StatsBomb xG")
axes[0].set_ylabel("Modelo propio xG")

examples = comparison.reindex(comparison["xg_diff_model_minus_statsbomb"].abs().sort_values(ascending=False).head(12).index)
sns.barplot(data=examples, y="player", x="xg_diff_model_minus_statsbomb", ax=axes[1], color="#0f172a")
axes[1].axvline(0, color="#94a3b8")
axes[1].set_title("Ejemplos con mayor diferencia de valoraci?n")
axes[1].set_xlabel("Modelo propio - StatsBomb xG")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

examples[["player", "team", "goal", "statsbomb_xg", "model_xg", "distance_to_goal", "shot_type", "shot_body_part", "under_pressure", "xg_diff_model_minus_statsbomb"]]

# 12. Insights futbol?sticos

A partir de las variables y predicciones aparecen varias lecturas:

1. La distancia domina gran parte de la se?al, pero no lo explica todo. Dos tiros a la misma distancia pueden tener probabilidades muy distintas si uno est? centrado y otro escorado.
2. La presi?n defensiva reduce la calidad del tiro. No solo importa llegar a zona de remate, sino llegar con tiempo y ventaja.
3. La variable `big_chance_custom` es simple, pero ?til como resumen de ocasiones claras. Es interpretable para t?cnicos y analistas.
4. La eficiencia ofensiva ayuda a separar volumen de finalizaci?n: un jugador puede generar mucho xG sin necesariamente convertir por encima de lo esperado.

In [ ]:
team_summary = (
    shots_test.groupby("team")
    .agg(shots=("goal", "size"), goals=("goal", "sum"), model_xg=("model_xg", "sum"), statsbomb_xg=("statsbomb_xg", "sum"))
    .assign(goal_minus_model_xg=lambda d: d["goals"] - d["model_xg"])
    .sort_values("model_xg", ascending=False)
    .reset_index()
)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=team_summary, x="model_xg", y="goals", size="shots", hue="goal_minus_model_xg", palette="coolwarm", sizes=(80, 400))
max_val = max(team_summary["model_xg"].max(), team_summary["goals"].max())
plt.plot([0, max_val], [0, max_val], "--", color="#94a3b8")
plt.title("Comparaci?n goles vs xG por equipo")
plt.xlabel("xG acumulado del modelo")
plt.ylabel("Goles")
plt.show()

team_summary

# 13. Conclusiones

El proyecto demuestra que un modelo de xG puede construirse de forma robusta con variables relativamente sencillas si est?n bien dise?adas desde el conocimiento del dominio.

La regresi?n log?stica ofrece una l?nea base interpretable. Random Forest y XGBoost capturan interacciones m?s complejas, especialmente entre posici?n, ?ngulo, zona y contexto. La evaluaci?n confirma que no basta con acertar goles: un buen xG debe ordenar y calibrar probabilidades.

El toque diferencial del proyecto est? en conectar t?cnica y f?tbol: mapas de tiro, heatmaps, comparaci?n con StatsBomb, ranking de eficiencia y ejemplos concretos donde el modelo discrepa de la referencia.

# 14. Limitaciones y trabajo futuro

Limitaciones:

- La simulaci?n reproduce estructura y l?gica, pero el valor real aparece con datos completos de StatsBomb.
- El modelo no incorpora posici?n de defensores ni portero, variables muy relevantes para xG avanzado.
- No se incluyen secuencias previas del ataque, velocidad de la jugada ni asistencia.
- La calibraci?n podr?a mejorarse con t?cnicas espec?ficas como Platt scaling o isotonic regression.

Trabajo futuro:

- Descargar StatsBomb Open Data real y seleccionar una competici?n concreta.
- A?adir contexto previo: tipo de asistencia, pase filtrado, contraataque o rebote.
- Entrenar modelos calibrados y comparar por competici?n.
- Crear una app sencilla para subir tiros y visualizar xG en tiempo real.